In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "SUIUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,3.2457,3.2473,3.2284,3.2319,108215.8,2025-06-01 00:04:59.999999+00:00,350407.18722,2174,47711.4,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,3.2319,3.2377,3.2304,3.2377,93634.3,2025-06-01 00:09:59.999999+00:00,302760.77169,1791,30952.9,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,0.000130,0.000072,0.000058,NaN,NaN
2,2025-06-01 00:10:00+00:00,3.2377,3.2377,3.2227,3.2249,65763.4,2025-06-01 00:14:59.999999+00:00,212271.21712,1911,27633.7,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000231,-0.000052,-0.000179,NaN,NaN
3,2025-06-01 00:15:00+00:00,3.2248,3.2269,3.2171,3.2258,134137.0,2025-06-01 00:19:59.999999+00:00,432099.25551,2121,77398.0,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000357,-0.000155,-0.000202,NaN,NaN
4,2025-06-01 00:20:00+00:00,3.2257,3.2320,3.2238,3.2301,60370.5,2025-06-01 00:24:59.999999+00:00,194907.18003,1533,32396.5,...,0.62349,0.201299,0.97953,1.224647e-16,-1.0,-0.000247,-0.000183,-0.000064,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,441
[info] optuna train rows: 53,401
[info] valid rows:        13,351
[info] test rows:         16,689


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-22 18:30:05,066] A new study created in memory with name: no-name-c9008daa-81ed-41af-91b7-ce67a6a7e51e


[I 2026-03-22 18:30:09,518] Trial 0 finished with value: 0.5302193040329537 and parameters: {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 0.5, 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 0 with value: 0.5302193040329537.


[I 2026-03-22 18:30:17,818] Trial 1 finished with value: 0.5222226037142059 and parameters: {'n_estimators': 500, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.8, 'bootstrap': False, 'class_weight': 'balanced_subsample', 'criterion': 'log_loss'}. Best is trial 0 with value: 0.5302193040329537.


[I 2026-03-22 18:30:21,393] Trial 2 finished with value: 0.5332959246782227 and parameters: {'n_estimators': 800, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 2 with value: 0.5332959246782227.


[I 2026-03-22 18:30:24,762] Trial 3 finished with value: 0.5333312463477753 and parameters: {'n_estimators': 700, 'max_depth': 12, 'min_samples_split': 11, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 3 with value: 0.5333312463477753.


...

[I 2026-03-22 18:33:38,591] Trial 96 finished with value: 0.5423189955181424 and parameters: {'n_estimators': 400, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': 'balanced_subsample', 'criterion': 'log_loss'}. Best is trial 96 with value: 0.5423189955181424.


[I 2026-03-22 18:33:41,946] Trial 97 pruned. 


[I 2026-03-22 18:33:44,210] Trial 98 finished with value: 0.5407851284550831 and parameters: {'n_estimators': 300, 'max_depth': 4, 'min_samples_split': 12, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': 'balanced_subsample', 'criterion': 'log_loss'}. Best is trial 96 with value: 0.5423189955181424.


[I 2026-03-22 18:33:47,191] Trial 99 finished with value: 0.5421029364111208 and parameters: {'n_estimators': 400, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': 'balanced_subsample', 'criterion': 'log_loss'}. Best is trial 96 with value: 0.5423189955181424.


['mom_60', 'mom_30', 'vol_30', 'vol_15', 'dist_ma_30', 'imbalance_15', 'vol_regime_ratio', 'macd_hist', 'range_15', 'atr_norm', 'trend_strength', 'imbalance_5', 'mom_10', 'range_5', 'dom_sin', 'dist_ma_15', 'mom_15', 'vol_ratio_5_30', 'range_ratio', 'vol_5', 'trend_x_imb', 'dist_ma_15_z', 'mom_5', 'mr_x_vol', 'hour_cos']
feature
mom_60              0.038416
mom_30              0.034958
vol_30              0.034847
vol_15              0.034765
dist_ma_30          0.034039
imbalance_15        0.033968
vol_regime_ratio    0.033794
macd_hist           0.032809
range_15            0.031533
atr_norm            0.031428
trend_strength      0.030867
imbalance_5         0.028856
mom_10              0.027938
range_5             0.027425
dom_sin             0.027388
dist_ma_15          0.026846
mom_15              0.026838
vol_ratio_5_30      0.026304
range_ratio         0.026282
vol_5               0.025121
trend_x_imb         0.024930
dist_ma_15_z        0.024780
mom_5               0.023425
mr

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

calibrator = artifacts["calibrator"]
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

In [11]:
X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

calibrator = artifacts["calibrator"]

train_pred = calibrator.predict_proba(X_train_full_sel)[:, 1]
test_pred = calibrator.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train ROC AUC:   0.560552
Test ROC AUC:    0.536694
Train PR AUC:    0.556540
Test PR AUC:     0.506053
Train Log Loss:  0.688019
Test Log Loss:   0.691569
Train Brier:     0.247451
Test Brier:      0.249207
Train Accuracy:  0.539250
Test Accuracy:   0.530649
Train Precision: 0.531057
Test Precision:  0.509468
Train Recall:    0.572430
Test Recall:     0.597803
Train F1:        0.550968
Test F1:         0.550112


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret": fwd_ret.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.428, 0.463] -0.000201   1669  0.007468
(0.463, 0.472] -0.000165   1669  0.006749
(0.472, 0.482] -0.000155   1669  0.007027
(0.482, 0.496] -0.000590   1669  0.006319
(0.496, 0.503] -0.000143   1669  0.006465
(0.503, 0.508] -0.000109   1668  0.006307
(0.508, 0.513] -0.000008   1669  0.006303
(0.513, 0.52]  -0.000039   1669  0.006492
(0.52, 0.532]  -0.000076   1669  0.007661
(0.532, 0.647]  0.000177   1669  0.010819


/tmp/ipykernel_1029515/1883822384.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret"].mean())
overall_mean_ret = float(eval_df["fwd_ret"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/SUIUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/SUIUSDT__h6_model.joblib
[saved] features -> models/rf/SUIUSDT__h6_feature_cols.json
[saved] feature importance -> models/rf/SUIUSDT__h6_feature_importance.csv
[saved] metadata -> models/rf/SUIUSDT__h6_meta.json
